In [2]:
%pip install pandas numpy scikit-learn joblib

  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.8 MB 6.3 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.8 MB 4.4 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.8 MB 4.4 MB/s eta 0:00:02
   --------- ------------------------------ 2.4/9.8 MB 3.0 MB/s eta 0:00:03
   ---------- ----------------------------- 2.6/9.8 MB 2.8 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/9.8 MB 2.4 MB/s eta 0:00:03
   ------------ --------------------------- 3.1/9.8 MB 2.4 MB/s eta 0:00:03
   -------------- ------------------------- 3.7/9.8 MB 2.2 MB/s et

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.11.0 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.11.0 requires cycler>=0.10, which is not installed.
matplotlib 3.11.0 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.11.0 requires kiwisolver>=1.3.1, which is not installed.
matplotlib 3.11.0 requires pillow>=9, which is not installed.
matplotlib 3.11.0 requires pyparsing>=3, which is not installed.


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score
import joblib

# Load Fused Historical Dataset
data_path = "Merged_dataset/fused_historical_database.csv"
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)
df

Loading data from Merged_dataset/fused_historical_database.csv...


,Merge_Year,Merge_Month,Country_Clean,Centroid_X,Centroid_Y,Duration_i,Dead,Displaced,Severity__,Affected_s,Magnitude,Country,ISO,Disaster Subtype,Total Deaths,No. Affected,Total Affected,Total Damage ('000 US$)
0,2010,8,niger,14.00,13.60,11.0,1.0,5000.0,1.0,619100.0,6.833153,Niger,NER,Riverine flood,3.0,226611.0,226611.0,NaN
1,2010,8,india,76.12,33.53,3.0,150.0,180.0,11.5,145000.0,6.699187,India,IND,Flash flood,196.0,12500.0,12725.0,NaN
2,2010,7,afghanistan,68.86,35.21,8.0,65.0,180.0,1.0,176800.0,6.150572,Afghanistan,AFG,Riverine flood,65.0,5000.0,5000.0,NaN
3,2010,7,pakistan,73.26,33.36,16.0,1600.0,14000000.0,2.0,129700.0,6.618090,Pakistan,PAK,Flash flood,1985.0,20356550.0,20359496.0,9500000.0
4,2010,7,pakistan,73.26,33.36,16.0,1600.0,14000000.0,2.0,129700.0,6.618090,Pakistan,PAK,Flash flood,60.0,4000.0,4000.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1543,2000,1,philippines,0.00,0.00,5.0,23.0,20000.0,1.0,16700.0,4.921686,Philippines,PHL,Flash flood,50.0,153885.0,153885.0,4080.0
1544,2000,1,philippines,0.00,0.00,5.0,23.0,20000.0,1.0,16700.0,4.921686,Philippines,PHL,Coastal flood,NaN,NaN,5250.0,NaN
1545,2000,1,mozambique,0.00,0.00,62.0,929.0,733000.0,1.5,436000.0,7.607969,Mozambique,MOZ,Riverine flood,800.0,4500000.0,4500000.0,419200.0
1546,2000,1,angola,0.00,0.00,8.0,31.0,70000.0,1.0,47000.0,5.575188,Angola,AGO,Riverine flood,31.0,70000.0,70000.0,10000.0


In [5]:
# Prepare Historical Flood Features (High & Medium Risk)
flood_samples = pd.DataFrame({
    'live_rain_mm': np.random.uniform(30.0, 150.0, size=len(df)),
    'humidity_pct': np.random.uniform(75.0, 100.0, size=len(df)),
    'wind_speed_ms': np.random.uniform(5.0, 25.0, size=len(df)),
    'past_floods_100km': np.random.randint(1, 15, size=len(df)),
    'nearest_flood_km': np.random.uniform(0.5, 45.0, size=len(df)),
    'risk_label': np.where(df['Severity__'] > 1.2, 2, 1) # 2: High, 1: Medium
})

# Synthesize Normal Weather Baseline (Low Risk - Class 0)
n_normal = len(df)
normal_samples = pd.DataFrame({
    'live_rain_mm': np.random.uniform(0.0, 10.0, size=n_normal),
    'humidity_pct': np.random.uniform(20.0, 65.0, size=n_normal),
    'wind_speed_ms': np.random.uniform(1.0, 10.0, size=n_normal),
    'past_floods_100km': np.random.choice([0, 1], size=n_normal, p=[0.8, 0.2]),
    'nearest_flood_km': np.random.uniform(60.0, 500.0, size=n_normal),
    'risk_label': 0 # 0: Low Risk
})

# Combine and Shuffle
dataset = pd.concat([flood_samples, normal_samples], ignore_index=True)
dataset = dataset.sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Total dataset shape: {dataset.shape}")

Total dataset shape: (3096, 6)


In [6]:
# Stratified Holdout Split (80% Train, 20% Test)
X = dataset.drop(columns=['risk_label'])
y = dataset['risk_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

# Train Random Forest Classifier
print("Training Risk Classifier...")
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
clf.fit(X_train, y_train)

# Evaluate Performance
y_pred = clf.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'Medium Risk', 'High Risk']))

Training Risk Classifier...
Accuracy: 90.00%

              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00       310
 Medium Risk       0.80      1.00      0.89       249
   High Risk       0.00      0.00      0.00        61

    accuracy                           0.90       620
   macro avg       0.60      0.67      0.63       620
weighted avg       0.82      0.90      0.86       620



In [ ]:
# Save the Trained Model
model_filename = "flood_model.pkl"
joblib.dump(clf, model_filename)
print(f"Model successfully exported to {model_filename}!")